# Exercise 7

Simulated annealing for TSP.

## Imports

Just normal setup first.

In [ ]:
import csv
import itertools
import math
import os
import random
import statistics
import time

try:
    import numpy as np
except ImportError:
    np = None

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

random.seed(12345)
PLOT_DIR = "Exercise7/pics"
ensure = os.makedirs
ensure(PLOT_DIR, exist_ok=True)


def save_fig(fig, filename):
    if plt is None:
        print("no matplotlib:", filename)
        return
    path = os.path.join(PLOT_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("saved", path)


print("numpy:", np is not None)
print("pandas:", pd is not None)
print("matplotlib:", plt is not None)


## Helpers

Need route cost, proposals, cooling schedules, all that stuff.

In [ ]:
def euclidean(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])


def make_distance_matrix(points):
    n = len(points)
    out = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                out[i][j] = euclidean(points[i], points[j])
    return out


def route_cost(route, cost_matrix):
    total = 0.0
    n = len(route)
    for i in range(n):
        a = route[i]
        b = route[(i + 1) % n]
        total += cost_matrix[a][b]
    return total


def random_route(n):
    route = list(range(n))
    random.shuffle(route)
    return route


def swap_two(route):
    new_route = route[:]
    i, j = random.sample(range(len(route)), 2)
    new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route


def reverse_segment(route):
    new_route = route[:]
    i, j = sorted(random.sample(range(len(route)), 2))
    new_route[i : j + 1] = reversed(new_route[i : j + 1])
    return new_route


def mixed_proposal(route):
    if random.random() < 0.5:
        return swap_two(route)
    return reverse_segment(route)


def schedule_sqrt(k, T0=1.0):
    return T0 / math.sqrt(1 + k)


def schedule_log(k, T0=1.0):
    return T0 / math.log(2 + k)


def schedule_geom(k, T0=1.0, alpha=0.9995):
    return T0 * (alpha ** k)


def propose(route, kind):
    if kind == "swap":
        return swap_two(route)
    if kind == "reverse":
        return reverse_segment(route)
    if kind == "mixed":
        return mixed_proposal(route)
    raise ValueError("unknown proposal")


def simulated_annealing(cost_matrix, initial_route, n_iter, schedule, proposal="swap"):
    current_route = initial_route[:]
    current_cost = route_cost(current_route, cost_matrix)
    best_route = current_route[:]
    best_cost = current_cost
    cost_history = [current_cost]
    best_history = [best_cost]
    accept_count = 0

    for k in range(1, n_iter + 1):
        candidate_route = propose(current_route, proposal)
        candidate_cost = route_cost(candidate_route, cost_matrix)
        delta = candidate_cost - current_cost
        temp = max(schedule(k), 1e-12)

        if delta <= 0 or random.random() < math.exp(-delta / temp):
            current_route = candidate_route
            current_cost = candidate_cost
            accept_count += 1

        if current_cost < best_cost:
            best_cost = current_cost
            best_route = current_route[:]

        cost_history.append(current_cost)
        best_history.append(best_cost)

    return {
        "best_route": best_route,
        "best_cost": best_cost,
        "cost_history": cost_history,
        "best_history": best_history,
        "acceptance_rate": accept_count / n_iter,
    }


def plot_route(points, route, title, filename):
    if plt is None:
        return
    xs = [points[i][0] for i in route] + [points[route[0]][0]]
    ys = [points[i][1] for i in route] + [points[route[0]][1]]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(xs, ys, "-o", markersize=4)
    for idx, (x, y) in enumerate(points):
        ax.text(x, y, str(idx + 1), fontsize=8)
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)
    save_fig(fig, filename)
    plt.close(fig)


def plot_history(cost_history, best_history, title, filename):
    if plt is None:
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(cost_history, alpha=0.5, label="current cost")
    ax.plot(best_history, linewidth=2, label="best so far")
    ax.set_title(title)
    ax.set_xlabel("iteration")
    ax.set_ylabel("cost")
    ax.grid(alpha=0.2)
    ax.legend()
    save_fig(fig, filename)
    plt.close(fig)


def circle_points(n, radius=1.0):
    pts = []
    for i in range(n):
        angle = 2 * math.pi * i / n
        pts.append((radius * math.cos(angle), radius * math.sin(angle)))
    return pts


def load_cost_matrix(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    if pd is not None:
        df = pd.read_csv(path, header=None)
        raw = df.values.tolist()
    else:
        with open(path, newline="") as f:
            raw = list(csv.reader(f))

    cleaned = []
    for row in raw:
        new_row = []
        for cell in row:
            if isinstance(cell, str):
                cell = cell.strip()
            if cell in ("", "-", None):
                new_row.append(0.0)
            else:
                new_row.append(float(cell))
        cleaned.append(new_row)

    ncols = max(len(row) for row in cleaned)
    cleaned = [row[:ncols] for row in cleaned if len(row) == ncols]
    return cleaned


def circle_layout(n):
    pts = []
    for i in range(n):
        angle = 2 * math.pi * i / n
        pts.append((math.cos(angle), math.sin(angle)))
    return pts


def plot_route_on_circle(route, title, filename):
    points = circle_layout(len(route))
    plot_route(points, route, title, filename)


def summarize_runs(rows, headers):
    widths = []
    for i, h in enumerate(headers):
        widths.append(max(len(h), max(len(str(r[i])) for r in rows)))
    print(" | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(row[i]).ljust(widths[i]) for i in range(len(headers))))


## Circle check

Starting with points on a circle bc then its easier to see if the method is doing something sensible.

In [ ]:
circle_pts = circle_points(20)
circle_matrix = make_distance_matrix(circle_pts)
circle_init = random_route(len(circle_pts))
circle_init_cost = route_cost(circle_init, circle_matrix)
plot_route(circle_pts, circle_init, "Circle - initial route", "circle_initial_route.png")

circle_result = simulated_annealing(
    circle_matrix,
    circle_init,
    n_iter=15000,
    schedule=lambda k: schedule_log(k, T0=1.2),
    proposal="reverse",
)

plot_route(circle_pts, circle_result["best_route"], "Circle - final route", "circle_final_route.png")
plot_history(
    circle_result["cost_history"],
    circle_result["best_history"],
    "Circle cost history",
    "cost_history_circle.png",
)

print("initial cost:", round(circle_init_cost, 4))
print("best cost:", round(circle_result["best_cost"], 4))
print("acceptance rate:", round(circle_result["acceptance_rate"], 4))
print("best route:", [x + 1 for x in circle_result["best_route"]])


If the final route mostly goes around the circle then the code is probly doing the right kind of thing.

## Random points

Then I tried random points in the plane, which is a bit more realistic and messier.

In [ ]:
random.seed(12345)
random_points = [(random.random(), random.random()) for _ in range(30)]
random_matrix = make_distance_matrix(random_points)
random_init = random_route(len(random_points))
random_init_cost = route_cost(random_init, random_matrix)
plot_route(random_points, random_init, "Random points - initial route", "random_points_initial_route.png")

random_runs = []
random_setups = [
    ("sqrt", lambda k: schedule_sqrt(k, T0=2.0)),
    ("log", lambda k: schedule_log(k, T0=2.0)),
]

for name, sched in random_setups:
    start = time.perf_counter()
    result = simulated_annealing(
        random_matrix,
        random_init,
        n_iter=25000,
        schedule=sched,
        proposal="reverse",
    )
    runtime = time.perf_counter() - start
    random_runs.append(
        {
            "schedule": name,
            "initial_cost": random_init_cost,
            "best_cost": result["best_cost"],
            "acceptance_rate": result["acceptance_rate"],
            "runtime": runtime,
            "result": result,
        }
    )

best_random = min(random_runs, key=lambda x: x["best_cost"])
plot_route(random_points, best_random["result"]["best_route"], "Random points - final route", "random_points_final_route.png")
plot_history(
    best_random["result"]["cost_history"],
    best_random["result"]["best_history"],
    "Random points cost history",
    "cost_history_random_points.png",
)

rows = []
for run in random_runs:
    rows.append([
        run["schedule"],
        round(run["initial_cost"], 3),
        round(run["best_cost"], 3),
        round(run["acceptance_rate"], 4),
        round(run["runtime"], 3),
    ])

summarize_runs(rows, ["schedule", "initial", "best", "accept rate", "runtime"])


In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for run in random_runs:
        ax.plot(run["result"]["best_history"], label=run["schedule"])
    ax.set_title("Cooling comparison on random points")
    ax.set_xlabel("iteration")
    ax.set_ylabel("best cost so far")
    ax.grid(alpha=0.2)
    ax.legend()
    save_fig(fig, "cooling_comparison.png")
    plt.close(fig)


The route gets way better than the random start, which is good enough here. I dont need to prove its optimal or anything.

## Small exact check

Did one tiny exact check too just so I wasnt fully guessing.

In [ ]:
small_pts = circle_points(8)
small_matrix = make_distance_matrix(small_pts)
small_init = random_route(8)
small_sa = simulated_annealing(
    small_matrix,
    small_init,
    n_iter=12000,
    schedule=lambda k: schedule_log(k, T0=1.0),
    proposal="reverse",
)

fixed_start = 0
others = list(range(1, 8))
exact_best_cost = float("inf")
exact_best_route = None
for perm in itertools.permutations(others):
    route = [fixed_start] + list(perm)
    cost = route_cost(route, small_matrix)
    if cost < exact_best_cost:
        exact_best_cost = cost
        exact_best_route = route

print("exact best cost:", round(exact_best_cost, 4))
print("sa best cost:", round(small_sa["best_cost"], 4))
print("difference:", round(small_sa["best_cost"] - exact_best_cost, 6))


## cost.csv

Now using the uploaded cost matrix. This part can be asymmetric so I didnt assume any symmetry.

In [ ]:
cost_matrix = load_cost_matrix("cost.csv")
print("shape:", len(cost_matrix), "x", len(cost_matrix[0]))
print("first rows:")
for row in cost_matrix[:5]:
    print(row[:8])

n_cost = len(cost_matrix)


In [ ]:
cost_schedules = {
    "sqrt": lambda k: schedule_sqrt(k, T0=220.0),
    "log": lambda k: schedule_log(k, T0=220.0),
    "geom": lambda k: schedule_geom(k, T0=220.0, alpha=0.9994),
}
proposals = ["swap", "reverse", "mixed"]

cost_runs = []
best_overall = None

for schedule_name, schedule_fn in cost_schedules.items():
    for proposal_name in proposals:
        for run_id in range(1, 5):
            initial = random_route(n_cost)
            start = time.perf_counter()
            result = simulated_annealing(
                cost_matrix,
                initial,
                n_iter=12000,
                schedule=schedule_fn,
                proposal=proposal_name,
            )
            runtime = time.perf_counter() - start
            row = {
                "run": run_id,
                "schedule": schedule_name,
                "proposal": proposal_name,
                "best_cost": result["best_cost"],
                "best_route": result["best_route"],
                "acceptance_rate": result["acceptance_rate"],
                "runtime": runtime,
                "result": result,
            }
            cost_runs.append(row)
            if best_overall is None or row["best_cost"] < best_overall["best_cost"]:
                best_overall = row

print("best overall cost:", round(best_overall["best_cost"], 3))
print("best route (1-based):", [x + 1 for x in best_overall["best_route"]])
print("acceptance rate:", round(best_overall["acceptance_rate"], 4))
print("schedule/proposal:", best_overall["schedule"], best_overall["proposal"])


In [ ]:
plot_history(
    best_overall["result"]["cost_history"],
    best_overall["result"]["best_history"],
    "Cost matrix history",
    "cost_matrix_cost_history.png",
)

plot_route_on_circle(
    best_overall["best_route"],
    "Cost matrix route on circle layout",
    "cost_matrix_best_route.png",
)


There are no real coordinates for this one, so the circle picture is only showing visit order. Its just for a rough visual.

## Comparison

Putting the schedule/proposal combos next to each other to see which ones seem better.

In [ ]:
summary = {}
for row in cost_runs:
    key = (row["schedule"], row["proposal"])
    summary.setdefault(key, []).append(row)

summary_rows = []
for key, rows_here in summary.items():
    best_cost = min(r["best_cost"] for r in rows_here)
    avg_best = statistics.mean(r["best_cost"] for r in rows_here)
    avg_accept = statistics.mean(r["acceptance_rate"] for r in rows_here)
    avg_runtime = statistics.mean(r["runtime"] for r in rows_here)
    summary_rows.append([
        key[0],
        key[1],
        len(rows_here),
        round(best_cost, 3),
        round(avg_best, 3),
        round(avg_accept, 4),
        round(avg_runtime, 3),
    ])

summary_rows.sort()
summarize_runs(
    summary_rows,
    ["schedule", "proposal", "runs", "best", "avg best", "avg accept", "avg runtime"],
)


In [ ]:
if plt is not None:
    proposal_names = ["swap", "reverse", "mixed"]
    fig, ax = plt.subplots(figsize=(7, 4))
    for proposal_name in proposal_names:
        xs = []
        ys = []
        for schedule_name in ["sqrt", "log", "geom"]:
            rows_here = summary[(schedule_name, proposal_name)]
            xs.append(schedule_name)
            ys.append(statistics.mean(r["best_cost"] for r in rows_here))
        ax.plot(xs, ys, marker="o", label=proposal_name)
    ax.set_title("Proposal comparison")
    ax.set_ylabel("average best cost")
    ax.grid(alpha=0.2)
    ax.legend()
    save_fig(fig, "proposal_comparison.png")
    plt.close(fig)


## Notes

- fast cooling can freeze too early
- slower cooling explores more but takes longer
- reverse moves help a lot for tsp-ish routes bc they can kill crossings
- multiple runs matter bc the result jumps around

## End

Overall SA does improve the tours quite a lot. The cost-matrix part depends a bunch on the cooling rule and the proposal, so doing several runs makes more sense than trusting one run.